In [1]:
# DEPENDENCIES

import sys
sys.path.insert(0, '..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

In [2]:
# SETUP

SUBJECTS = helper_functions.fuglsang_get_subjects()
N = len(SUBJECTS)

In [3]:
# HELPER FUNCTIONS

def report_binomial(label, correct, n, chance=0.5):
    result = binomtest(int(correct), n, chance, alternative='greater')
    print(f"{label}")
    print(f"  Accuracy: {correct}/{n} ({correct/n:.1%})")
    print(f"  Binomial test vs chance: p = {result.pvalue:.4f}"
          + (" *" if result.pvalue < 0.05 else ""))
    print()

def report_ttest(label, r_att, r_ign):
    r_att = np.array(list(r_att.values()) if isinstance(r_att, dict) else r_att)
    r_ign = np.array(list(r_ign.values()) if isinstance(r_ign, dict) else r_ign)
    t, p = ttest_rel(r_att, r_ign)
    d = (r_att - r_ign).mean() / (r_att - r_ign).std()
    print(f"{label}")
    print(f"  r_att: {r_att.mean():.4f} ± {r_att.std():.4f}")
    print(f"  r_ign: {r_ign.mean():.4f} ± {r_ign.std():.4f}")
    print(f"  Paired t-test: t = {t:.3f}, p = {p:.4f}"
          + (" *" if p < 0.05 else ""))
    print(f"  Cohen's d = {d:.3f}")
    print()

def report_mcnemar(label, decisions_a, decisions_b):
    da = np.array(list(decisions_a.values()) if isinstance(decisions_a, dict) else decisions_a)
    db = np.array(list(decisions_b.values()) if isinstance(decisions_b, dict) else decisions_b)
    table = np.array([
        [np.sum( da &  db), np.sum( da & ~db)],
        [np.sum(~da &  db), np.sum(~da & ~db)]
    ])
    result = mcnemar(table, exact=True)
    print(f"{label}")
    print(f"  Contingency table:\n    {table[0].tolist()}\n    {table[1].tolist()}")
    print(f"  McNemar test: p = {result.pvalue:.4f}"
          + (" *" if result.pvalue < 0.05 else ""))
    print()

In [5]:
#RUN STATISTICS

# ─────────────────────────────────────────────────────────────────
# RUN CLASSIFIERS
# ─────────────────────────────────────────────────────────────────

testing_subjects = SUBJECTS[-5:]   # S14-S18

print("Running hold-out classifiers...")
acc_ho_env,   dec_ho_env,   r_att_ho_env,   r_ign_ho_env   = helper_functions.aad_classifier(
    PREDICTOR_TYPE.ENVELOPE,       testing_subjects,
    generalised=GENERALISATION_TYPE.AVERAGE, cv=CROSS_VALIDATION_TYPE.HOLD_OUT,
    aad_type=AAD_APPROACH.DOUBLE
)
acc_ho_onset, dec_ho_onset, r_att_ho_onset, r_ign_ho_onset = helper_functions.aad_classifier(
    PREDICTOR_TYPE.ENVELOPE_ONSET, testing_subjects,
    generalised=GENERALISATION_TYPE.AVERAGE, cv=CROSS_VALIDATION_TYPE.HOLD_OUT,
    aad_type=AAD_APPROACH.DOUBLE
)

print("\nRunning LOOCV classifiers...")
acc_loo_env,   dec_loo_env,   r_att_loo_env,   r_ign_loo_env   = helper_functions.aad_classifier(
    PREDICTOR_TYPE.ENVELOPE,       SUBJECTS,
    generalised=GENERALISATION_TYPE.AVERAGE, cv=CROSS_VALIDATION_TYPE.LOO,
    aad_type=AAD_APPROACH.DOUBLE
)
acc_loo_onset, dec_loo_onset, r_att_loo_onset, r_ign_loo_onset = helper_functions.aad_classifier(
    PREDICTOR_TYPE.ENVELOPE_ONSET, SUBJECTS,
    generalised=GENERALISATION_TYPE.AVERAGE, cv=CROSS_VALIDATION_TYPE.LOO,
    aad_type=AAD_APPROACH.DOUBLE
)

print("\nRunning pooled LOOCV classifiers...")
acc_pool_env,   dec_pool_env,   r_att_pool_env,   r_ign_pool_env   = helper_functions.aad_classifier(
    PREDICTOR_TYPE.ENVELOPE,       SUBJECTS,
    generalised=GENERALISATION_TYPE.POOLED, cv=CROSS_VALIDATION_TYPE.LOO,
    aad_type=AAD_APPROACH.DOUBLE
)
acc_pool_onset, dec_pool_onset, r_att_pool_onset, r_ign_pool_onset = helper_functions.aad_classifier(
    PREDICTOR_TYPE.ENVELOPE_ONSET, SUBJECTS,
    generalised=GENERALISATION_TYPE.POOLED, cv=CROSS_VALIDATION_TYPE.LOO,
    aad_type=AAD_APPROACH.SINGLE
)

# ─────────────────────────────────────────────────────────────────
# 1. BINOMIAL TESTS
# ─────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("BINOMIAL TESTS — Classification accuracy vs chance (50%)")
print("=" * 60 + "\n")

report_binomial("Hold-out — Envelope",
                sum(dec_ho_env.values()), len(testing_subjects))
report_binomial("Hold-out — Onset",
                sum(dec_ho_onset.values()), len(testing_subjects))
report_binomial("LOOCV — Envelope",
                sum(dec_loo_env.values()), N)
report_binomial("LOOCV — Onset",
                sum(dec_loo_onset.values()), N)
report_binomial("Pooled LOOCV — Envelope",
                sum(dec_pool_env.values()), N)
report_binomial("Pooled LOOCV — Onset",
                sum(dec_pool_onset.values()), N)

# ─────────────────────────────────────────────────────────────────
# 2. PAIRED T-TESTS
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("PAIRED T-TESTS — r_attended vs r_ignored")
print("=" * 60 + "\n")

report_ttest("Hold-out — Envelope",      r_att_ho_env,    r_ign_ho_env)
report_ttest("Hold-out — Onset",         r_att_ho_onset,  r_ign_ho_onset)
report_ttest("LOOCV — Envelope",         r_att_loo_env,   r_ign_loo_env)
report_ttest("LOOCV — Onset",            r_att_loo_onset, r_ign_loo_onset)
report_ttest("Pooled LOOCV — Envelope",  r_att_pool_env,  r_ign_pool_env)
report_ttest("Pooled LOOCV — Onset",     r_att_pool_onset,r_ign_pool_onset)

# ─────────────────────────────────────────────────────────────────
# 3. McNEMAR TESTS
# ─────────────────────────────────────────────────────────────────

print("=" * 60)
print("McNEMAR TESTS — Model comparisons")
print("=" * 60 + "\n")

report_mcnemar("Hold-out: Envelope vs Onset",
               dec_ho_env,   dec_ho_onset)
report_mcnemar("LOOCV: Envelope vs Onset",
               dec_loo_env,  dec_loo_onset)
report_mcnemar("Pooled LOOCV: Envelope vs Onset",
               dec_pool_env, dec_pool_onset)
report_mcnemar("LOOCV: Averaged vs Pooled (Envelope)",
               dec_loo_env,  dec_pool_env)
report_mcnemar("LOOCV: Averaged vs Pooled (Onset)",
               dec_loo_onset,dec_pool_onset)



Running hold-out classifiers...
S14: r_att=0.175, r_ign=0.068
S15: r_att=0.202, r_ign=0.077
S16: r_att=0.096, r_ign=0.086
S17: r_att=0.088, r_ign=0.010
S18: r_att=0.148, r_ign=0.043

✅ Classification rate (PREDICTOR_TYPE.ENVELOPE): 100.00%

────────────────────────────────────────────────────────────

S14: r_att=0.077, r_ign=0.033
S15: r_att=0.087, r_ign=0.035
S16: r_att=0.051, r_ign=0.034
S17: r_att=0.035, r_ign=0.000
S18: r_att=0.059, r_ign=0.023

✅ Classification rate (PREDICTOR_TYPE.ENVELOPE_ONSET): 100.00%

────────────────────────────────────────────────────────────


Running LOOCV classifiers...
S1: r_att=0.044, r_ign=0.030
S2: r_att=0.137, r_ign=0.083
S3: r_att=0.080, r_ign=0.045
S4: r_att=0.148, r_ign=0.090
S5: r_att=0.117, r_ign=0.045
S6: r_att=0.055, r_ign=0.020
S7: r_att=0.222, r_ign=0.076
S8: r_att=0.099, r_ign=0.045
S9: r_att=0.110, r_ign=0.055
S10: r_att=0.091, r_ign=0.026
S11: r_att=0.005, r_ign=0.005
S12: r_att=0.064, r_ign=0.067
S13: r_att=0.134, r_ign=0.070
S14: r_at

In [9]:
# COMPARE POOLED UNIVERSAL TRF TO AVERAGE UNIVERSAL TRF DIRECTLY (CORRELATION MEASURE)

# ─────────────────────────────────────────────
# Load both TRFs
# ─────────────────────────────────────────────

models_to_compare = [
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED),
    (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED),
    (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED),
]

print("=" * 60)
print("TRF SIMILARITY — Averaged vs Pooled universal TRFs")
print("=" * 60 + "\n")

for predictor, attention in models_to_compare:

    avg_name  = helper_functions.get_trf_model_name(
        DATASET_TYPE.FUGLSANG, predictor, attention,
        MODEL_TYPE.BACKWARD, GENERALISATION_TYPE.AVERAGE
    )
    pool_name = helper_functions.get_trf_model_name(
        DATASET_TYPE.FUGLSANG, predictor, attention,
        MODEL_TYPE.BACKWARD, GENERALISATION_TYPE.POOLED
    )

    trf_avg  = eelbrain.load.unpickle(
        FUGLSANG_GENERAL_TRF_DIR / f'full_{avg_name}.pickle'
    ).h_scaled
    trf_pool = eelbrain.load.unpickle(
        FUGLSANG_GENERAL_TRF_DIR / f'full_{pool_name}.pickle'
    ).h_scaled

    # Flatten weight matrices to 1D vectors for correlation
    w_avg  = trf_avg.x.flatten()
    w_pool = trf_pool.x.flatten()

    # Pearson correlation between weight matrices
    r = np.corrcoef(w_avg, w_pool)[0, 1]

    # L2 norm of the difference (normalised by L2 norm of averaged TRF)
    # Included for completeness but less interpretable
    l2_diff      = np.linalg.norm(w_avg - w_pool)
    l2_avg       = np.linalg.norm(w_avg)
    l2_relative  = l2_diff / l2_avg

    label = f"{predictor.value} — {attention.value}"
    print(f"{label}")
    print(f"  Pearson r (weight matrices):    {r:.4f}")
    print(f"  L2 norm difference (relative):  {l2_relative:.4f}")
    print()

TRF SIMILARITY — Averaged vs Pooled universal TRFs

envelope — attended
  Pearson r (weight matrices):    0.6444
  L2 norm difference (relative):  1.4260

envelope — ignored
  Pearson r (weight matrices):    0.5518
  L2 norm difference (relative):  1.8317

envelope_onset — attended
  Pearson r (weight matrices):    0.6081
  L2 norm difference (relative):  1.6216

envelope_onset — ignored
  Pearson r (weight matrices):    0.4730
  L2 norm difference (relative):  2.0886

